# Lekshon 09 - Design Patɛn wey dey help you sabi how you dey think


## How to set am up

Dis notebook dey demonstrate the Metacognition design pattern using the Microsoft Agent Framework.

**Wetin you need:**
- Azure OpenAI deployment wey don configure via environment variables
- Azure CLI don authenticate (`az login`)


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Wetin be Metacognition?

Metacognition na **to think about how you dey think**. For di context of AI agents, e mean say make you build agents wey fit:

- **Dey reflect** on dem own outputs and how dem reason
- **Find errors** and recover well instead of just dey fail quietly
- **Check** if dem responses complete and helpful
- **Adapt** dem strategy when di first approach no work (for example, fall back to a backup system)

A metacognitive agent no just dey answer questions — e dey monitor im own performance and e dey adjust as e dey go.


## Main an Backup Tools

One common metacognition pattern na the **fallback strategy**. Di agent go try di primary tool first; if e fail (e.g., a 404 error), di agent go notice di failure an openly switch to di backup tool.

E mirror real-world systems wey primary services fit no dey available an agents gats self-diagnose di issue before dem choose an alternative path.

Below we go define two flight lookup tools:
- **Primary** — e dey cover Paris, Tokyo, and Barcelona
- **Backup** — e dey cover Berlin, Sydney, and New York City


In [ ]:
@tool(approval_mode="never_require")
def get_flight_times(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times for a destination (primary source)."""
    flights = {
        "Paris": "Departures: 08:00, 12:30, 17:45 — from $350",
        "Tokyo": "Departures: 11:00, 23:30 — from $890",
        "Barcelona": "Departures: 07:15, 14:00, 19:30 — from $280",
    }
    if destination in flights:
        return flights[destination]
    raise Exception(f"404: No flights found for {destination} in primary system")


@tool(approval_mode="never_require")
def get_flight_times_backup(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times from backup system (used when primary fails)."""
    backup_flights = {
        "Berlin": "Departures: 09:00, 16:00 — from $220",
        "Sydney": "Departures: 22:00 — from $1200",
        "New York City": "Departures: 06:00, 10:30, 15:00, 20:00 — from $450",
    }
    return backup_flights.get(
        destination,
        f"No flights found for {destination} in any system. Please try again later.",
    )

## Agent wey dey self-check wey fit recover from error

Di agent wey dey below don get instruction make e try di primary flight system first, sabi when e fail, an openly switch to di backup system. After each response, e go do small self-check to see if e don fully answer di person wey ask.


In [ ]:
agent = await provider.create_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="FlightBookingAgent",
    instructions="""You are a flight booking agent with self-reflection capabilities.

When looking up flights:
1. Try the primary flight system first (get_flight_times)
2. If the primary system fails (404 error), acknowledge the error and try the backup system (get_flight_times_backup)
3. Always explain to the user what happened — be transparent about fallbacks
4. If both systems fail, apologize and suggest alternatives

After each response, briefly evaluate whether your answer was complete and helpful.""",
)

# Test with a destination in primary system
print("=== Test 1: Destination in primary system ===")
response = await agent.run(
    "What flights are available to Paris?",
    )
print(response)

# Test with a destination only in backup system
print("\n=== Test 2: Destination only in backup system ===")
response = await agent.run(
    "What flights are available to Berlin?",
    )
print(response)

## Patern for Self-Check

Another side of metacognition na **self-check**: another agent (or the same agent wey go do am again) go review di response to make sure say e complete, correct, and helpful.

Below we go create a `ResponseEvaluator` agent wey go score travel-agent responses for three areas.


In [ ]:
evaluation_agent = await provider.create_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="ResponseEvaluator",
    instructions="""You are a quality evaluator for travel agent responses.
Given a travel question and the agent's response, evaluate:
1. Completeness: Did it answer all parts of the question? (1-5)
2. Accuracy: Is the information correct? (1-5)
3. Helpfulness: Would a traveler find this useful? (1-5)
Provide a brief evaluation with scores and one suggestion for improvement.""",
)

# Evaluate the agent's response from Test 1
eval_prompt = f"""Question: What flights are available to Paris?
Agent Response: {response}

Please evaluate the above response."""

evaluation = await evaluation_agent.run(eval_prompt)
print("=== Self-Evaluation ===")
print(evaluation)

## Summary

In this lesson you learned how to build **agents wey dey reflect on dia own thinking** using the Microsoft Agent Framework:

- **Self-reflection**: Agents wey dey monitor dia own reasoning and dey transparently communicate wetin happen.
- **Error recovery with fallbacks**: Na primary + backup tool pattern where the agent detects failures (e.g., 404 errors) and automatically tries an alternative source.
- **Self-evaluation**: A separate evaluator agent wey dey score responses for how complete dem be, how correct dem be, and how dem dey helpful.

These patterns make agents more robust, transparent, and trustworthy — na critical qualities for production deployments.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
Disclaimer:
Dis document don translate by AI translation service Co-op Translator (https://github.com/Azure/co-op-translator). Even though we dey try make am correct, abeg note say automatic translations fit get errors or wrong tins. Di original document for im own language na di official source wey you suppose trust. For important mata, make you use professional human translator. We no dey responsible or liable for any misunderstanding or wrong interpretation wey fit come from dis translation.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
